# Phase 3 -- Before/after evaluation: base Qwen3-8B vs. fine-tuned

Runs on **Colab or Kaggle free-tier GPU only** (same hardware contract as Phase 2 -- this machine never runs model inference locally except the final quantized GGUF in Phase 4). This is the core portfolio evidence for the project: does the QLoRA + DoRA adapter actually help on structured extraction, measured over the full 140-example held-out eval set from `data/processed/eval.jsonl`.

**Flagged assumptions:**
1. **Decoding is greedy (`do_sample=False`)**, not the temperature=0.7/top_p=0.8/top_k=20 sampling Qwen3's docs suggest for general chat quality. Chosen deliberately for a controlled before/after comparison -- sampling would make the two models' outputs (and any rerun of this notebook) non-reproducible, which matters more here than matching general chat-quality recommendations for a structured-extraction accuracy eval.
2. **Examples run one at a time, not batched.** Slower, but avoids left-padding/attention-mask edge cases that batched causal-LM generation can get subtly wrong -- and accuracy matters more than speed here.

`FastLanguageModel.from_pretrained` pointed at a local adapter directory is now confirmed working -- this notebook has been run successfully twice (v1 and v2 adapters) as of the last retrain round. The commented-out `PeftModel.from_pretrained` fallback cell is kept below just in case a future adapter version hits an edge case, but it's no longer an open risk.

## Upload checklist

Bundle both of these into **one** Kaggle Dataset -- any dataset name works, the notebook auto-detects it by scanning `/kaggle/input/*/` for the right files, so no path editing is needed after re-uploading:

- `data/processed/eval.jsonl` (140 rows, never changes between rounds)
- Only the **inference-relevant** adapter files (not the whole download): `adapter_config.json`, `adapter_model.safetensors`, `tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja`, `README.md`. Skip `optimizer.pt`/`scheduler.pt`/`scaler.pt`/`rng_state.pth`/`training_args.bin` -- those are training-resumption state (~104MB), not needed for inference.

In [ ]:
%%capture
!pip install unsloth

## 1. Load the eval set

In [ ]:
import glob, json, os, shutil

def find_kaggle_dataset(*required_files):
    """Search /kaggle/input/*/ for a folder containing all of required_files --
    works no matter what the attached Dataset is named, so re-uploading under a new
    name never requires editing this notebook."""
    for candidate in sorted(glob.glob("/kaggle/input/*/")):
        if all(os.path.exists(os.path.join(candidate, f)) for f in required_files):
            return candidate.rstrip("/")
    return None

if not os.path.exists("eval.jsonl"):
    kaggle_dir = find_kaggle_dataset("eval.jsonl")
    if kaggle_dir:
        shutil.copy(os.path.join(kaggle_dir, "eval.jsonl"), "eval.jsonl")
        print(f"found and copied eval.jsonl from {kaggle_dir}")
    else:
        try:
            from google.colab import files
            print("Upload data/processed/eval.jsonl:")
            files.upload()
        except ImportError:
            raise RuntimeError(
                "eval.jsonl not found locally, and no /kaggle/input/*/ folder contains it. "
                "On Kaggle: attach a Dataset containing eval.jsonl -- any dataset name works, "
                "no path editing needed."
            )

with open("eval.jsonl", encoding="utf-8") as f:
    eval_rows = [json.loads(line) for line in f]
print(f"eval examples: {len(eval_rows)}")
assert len(eval_rows) == 140, f"expected 140 eval rows, got {len(eval_rows)} -- check the uploaded file"

## 2. Shared prompt format and JSON parsing

`SYSTEM_PROMPT` copied verbatim from `notebooks/train_qlora_dora.ipynb` -- must match training exactly, since a different system prompt at eval time would confound the before/after comparison for the fine-tuned model.

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are an automotive safety complaint analyst. Given a raw consumer complaint "
    "about a vehicle, extract a structured JSON object with exactly these fields: "
    'component (string), defect_type (string), safety_risk ("yes" or "no"), '
    'severity ("low", "medium", or "high"). Respond with only the JSON object.'
)

MAX_SEQ_LENGTH = 768  # same as training
MAX_NEW_TOKENS = 100  # the target JSON is ~30-50 tokens; generous headroom without wasting compute

def build_prompt(tokenizer, narrative):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Complaint:\n{narrative}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

_JSON_OBJ_PATTERN = re.compile(r"\{.*?\}", re.DOTALL)

def parse_json_output(raw_text):
    """Extract and parse the first {...} block. Returns (parsed_dict_or_None, raw_text).
    Never raises -- a model going off-script (extra prose, missing braces, etc.) is a
    JSON-validity failure to count, not a notebook crash."""
    match = _JSON_OBJ_PATTERN.search(raw_text)
    if not match:
        return None, raw_text
    try:
        obj = json.loads(match.group(0))
        if not isinstance(obj, dict):
            return None, raw_text
        return obj, raw_text
    except json.JSONDecodeError:
        return None, raw_text

## 3. Generation loop (shared by both models)

In [ ]:
import torch
from unsloth import FastLanguageModel

def run_eval(model, tokenizer, rows, label):
    FastLanguageModel.for_inference(model)  # Unsloth's native 2x-faster inference mode
    results = []
    for i, row in enumerate(rows):
        prompt_text = build_prompt(tokenizer, row["narrative"])
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,          # greedy -- see flagged assumption #1 above
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)
        parsed, raw = parse_json_output(raw_output)
        results.append({"odino": row["odino"], "parsed": parsed, "raw_output": raw})
        if (i + 1) % 20 == 0 or (i + 1) == len(rows):
            print(f"[{label}] {i + 1}/{len(rows)}")
    return results

## 4. Run the base model (zero-shot, no adapter)

In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,
)

base_results = run_eval(base_model, base_tokenizer, eval_rows, "base")

## 5. Free GPU memory before loading the fine-tuned model

Loading both 8B (4-bit) models simultaneously would roughly double VRAM use for no reason -- evaluate sequentially instead.

In [ ]:
import gc

del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()

## 6. Run the fine-tuned model (base + LoRA/DoRA adapter)

In [ ]:
ADAPTER_DIR = find_kaggle_dataset("adapter_config.json", "adapter_model.safetensors")
if ADAPTER_DIR is None:
    raise RuntimeError(
        "No /kaggle/input/*/ folder found containing adapter_config.json + "
        "adapter_model.safetensors. Attach a Dataset with the adapter's inference files "
        "in it (see the upload checklist at the top) -- any dataset name works, no path "
        "editing needed."
    )
print(f"using adapter from: {ADAPTER_DIR}")

finetuned_model, finetuned_tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,
)

finetuned_results = run_eval(finetuned_model, finetuned_tokenizer, eval_rows, "finetuned")

In [ ]:
# Fallback if the cell above fails to auto-attach the adapter (flagged assumption #2):
# from peft import PeftModel
# finetuned_model, finetuned_tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit", max_seq_length = MAX_SEQ_LENGTH,
#     load_in_4bit = True, dtype = None,
# )
# finetuned_model = PeftModel.from_pretrained(finetuned_model, ADAPTER_DIR)
# finetuned_results = run_eval(finetuned_model, finetuned_tokenizer, eval_rows, "finetuned")

## 7. Metrics

In [ ]:
def norm(s):
    return (s or "").strip().upper() if isinstance(s, str) else ""

def field_correct(pred, actual_row, field):
    if pred is None:
        return False
    return norm(pred.get(field)) == norm(actual_row[field])

def compute_metrics(results, rows):
    n = len(rows)
    preds = [r["parsed"] for r in results]

    json_valid = sum(1 for p in preds if p is not None)
    metrics = {
        "n": n,
        "json_validity_rate": json_valid / n,
        "component_accuracy": sum(field_correct(p, r, "component") for p, r in zip(preds, rows)) / n,
        "defect_type_accuracy": sum(field_correct(p, r, "defect_type") for p, r in zip(preds, rows)) / n,
        "safety_risk_accuracy": sum(field_correct(p, r, "safety_risk") for p, r in zip(preds, rows)) / n,
        "severity_accuracy": sum(field_correct(p, r, "severity") for p, r in zip(preds, rows)) / n,
    }

    # safety_risk=yes precision/recall -- the single most important number (Section 3 PM framing)
    tp = sum(1 for p, r in zip(preds, rows) if r["safety_risk"] == "yes" and p is not None and norm(p.get("safety_risk")) == "YES")
    fp = sum(1 for p, r in zip(preds, rows) if r["safety_risk"] == "no" and p is not None and norm(p.get("safety_risk")) == "YES")
    fn = sum(1 for p, r in zip(preds, rows) if r["safety_risk"] == "yes" and not (p is not None and norm(p.get("safety_risk")) == "YES"))
    metrics["safety_risk_yes_precision"] = tp / (tp + fp) if (tp + fp) > 0 else None
    metrics["safety_risk_yes_recall"] = tp / (tp + fn) if (tp + fn) > 0 else None
    metrics["safety_risk_yes_tp_fp_fn"] = {"tp": tp, "fp": fp, "fn": fn}

    # severity confusion matrix: actual (rows) x predicted (columns), predicted includes INVALID
    sev_labels = ["low", "medium", "high"]
    matrix = {a: {p: 0 for p in sev_labels + ["INVALID"]} for a in sev_labels}
    for p, r in zip(preds, rows):
        actual = r["severity"]
        pred_sev = norm(p.get("severity")).lower() if p is not None else None
        if pred_sev not in sev_labels:
            pred_sev = "INVALID"
        matrix[actual][pred_sev] += 1
    metrics["severity_confusion_matrix"] = matrix
    metrics["severity_per_tier_accuracy"] = {
        tier: matrix[tier][tier] / sum(matrix[tier].values()) if sum(matrix[tier].values()) > 0 else None
        for tier in sev_labels
    }
    return metrics

base_metrics = compute_metrics(base_results, eval_rows)
finetuned_metrics = compute_metrics(finetuned_results, eval_rows)

In [ ]:
def pct(x):
    return f"{x:.1%}" if x is not None else "n/a"

print(f"{'metric':<28} {'base':>12} {'fine-tuned':>12}")
for key, label in [
    ("json_validity_rate", "JSON validity rate"),
    ("component_accuracy", "component accuracy"),
    ("defect_type_accuracy", "defect_type accuracy"),
    ("safety_risk_accuracy", "safety_risk accuracy"),
    ("severity_accuracy", "severity accuracy"),
]:
    print(f"{label:<28} {pct(base_metrics[key]):>12} {pct(finetuned_metrics[key]):>12}")

print()
print("safety_risk=yes precision/recall (the number that matters most for the safety-triage story):")
print(f"{'':<28} {'base':>12} {'fine-tuned':>12}")
print(f"{'precision':<28} {pct(base_metrics['safety_risk_yes_precision']):>12} {pct(finetuned_metrics['safety_risk_yes_precision']):>12}")
print(f"{'recall':<28} {pct(base_metrics['safety_risk_yes_recall']):>12} {pct(finetuned_metrics['safety_risk_yes_recall']):>12}")

print()
print("severity per-tier accuracy:")
print(f"{'':<10} {'base':>12} {'fine-tuned':>12}")
for tier in ["low", "medium", "high"]:
    print(f"{tier:<10} {pct(base_metrics['severity_per_tier_accuracy'][tier]):>12} {pct(finetuned_metrics['severity_per_tier_accuracy'][tier]):>12}")

print()
print("fine-tuned severity confusion matrix (rows=actual, cols=predicted):")
print(f"{'':<8}" + "".join(f"{c:>9}" for c in ["low", "medium", "high", "INVALID"]))
for a in ["low", "medium", "high"]:
    row = finetuned_metrics["severity_confusion_matrix"][a]
    print(f"{a:<8}" + "".join(f"{row[c]:>9}" for c in ["low", "medium", "high", "INVALID"]))

## 8. Failure examples from the fine-tuned model

Up to 8 examples where the fine-tuned model got at least one field wrong. `guess_why` is a
data-grounded heuristic (checks the raw multi-component metadata and safety flags actually
present in the eval row), not a fabricated explanation -- treat it as a starting point for
the real error-analysis writeup in `docs/eval-report.md`, not a final verdict.

In [ ]:
def guess_why(row, pred):
    if pred is None:
        return "model did not produce parseable JSON"
    notes = []
    if not field_correct(pred, row, "component"):
        raw = row.get("component_raw", "")
        if "," in raw or ":" in raw:
            notes.append(f"multi/hierarchical component ('{raw}') -- model may have picked a different one than the first-listed primary label")
        else:
            notes.append("component mismatch on a single-component complaint -- possible taxonomy ambiguity")
    if not field_correct(pred, row, "defect_type"):
        notes.append("defect_type mismatch -- narrative may span more than one plausible defect category")
    if not field_correct(pred, row, "safety_risk"):
        direction = "missed a real safety signal" if row["safety_risk"] == "yes" else "over-flagged a non-risk complaint"
        notes.append(f"safety_risk miss (crash={row['crash']} fire={row['fire']} injured={row['injured']} deaths={row['deaths']}) -- {direction}")
    if not field_correct(pred, row, "severity"):
        notes.append("severity tier mismatch")
    return "; ".join(notes) if notes else "no field mismatch detected (check formatting/whitespace)"

failure_examples = []
for row, result in zip(eval_rows, finetuned_results):
    pred = result["parsed"]
    is_failure = pred is None or any(
        not field_correct(pred, row, f) for f in ["component", "defect_type", "safety_risk", "severity"]
    )
    if is_failure:
        failure_examples.append({
            "odino": row["odino"],
            "narrative": row["narrative"],
            "predicted": pred,
            "raw_output": result["raw_output"] if pred is None else None,
            "actual": {
                "component": row["component"], "defect_type": row["defect_type"],
                "safety_risk": row["safety_risk"], "severity": row["severity"],
            },
            "guess_why": guess_why(row, pred),
        })

failure_sample = failure_examples[:8]
print(f"{len(failure_examples)} total failing examples; showing {len(failure_sample)}\n")
for ex in failure_sample:
    print(f"odino={ex['odino']}")
    print(f"narrative: {ex['narrative'][:200]}")
    print(f"predicted: {ex['predicted']}")
    print(f"actual:    {ex['actual']}")
    print(f"guess why: {ex['guess_why']}")
    print()

## 9. Save results

In [ ]:
output = {
    "eval_set_size": len(eval_rows),
    "generation_config": {
        "do_sample": False,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "enable_thinking": False,
    },
    "base": {
        "model": "unsloth/Qwen3-8B-unsloth-bnb-4bit (no adapter)",
        "metrics": base_metrics,
        "predictions": base_results,
    },
    "finetuned": {
        # dynamic, not a hardcoded epoch/eval_loss string -- that string went stale twice
        # across the v1/v2 reruns and had to be hand-corrected after the fact. The actual
        # adapter identity (which epoch, which eval_loss) lives in
        # docs/training-hyperparameters.md, cross-referenced by this ADAPTER_DIR path.
        "model": f"unsloth/Qwen3-8B-unsloth-bnb-4bit + adapter from {ADAPTER_DIR}",
        "metrics": finetuned_metrics,
        "predictions": finetuned_results,
        "failure_examples": failure_examples,
    },
}

with open("eval_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print("saved eval_results.json -- download this and place it at eval/eval_results.json in the repo")

try:
    from google.colab import files
    files.download("eval_results.json")
except ImportError:
    print("Not on Colab -- grab eval_results.json from the Kaggle notebook's output files panel instead.")

## Next

Bring `eval_results.json` back to the repo (`eval/eval_results.json`) for the error-analysis
writeup in `docs/eval-report.md`. Results are reported here exactly as computed -- including
if the fine-tuned model is worse than base on some metric. That's a real finding to write up
honestly, not something to obscure.